# AskSight — model training (Kaggle / Colab GPU)

`photo → CLIP (frozen) → connector (trained) → Qwen2.5-1.5B (frozen) → answer text`

## Kaggle quick start

1. Push the current AskSight changes to the `feat/multimodal-core-workflow` branch. This notebook clones that branch to get the training code.
2. In Kaggle, create a Notebook, upload this `.ipynb`, then open **Settings** and set **Accelerator → GPU** and **Internet → On**.
3. Run all cells. The default is 20,000 VQA training questions. Download the checkpoint zip from the notebook's **Output** panel before ending the session.

Estimated wall time: allow **4–7 hours** for the 20k run, including data setup. **50k may take about 8–16 hours** (a rough scale-up estimate, not measured); it may exceed Kaggle's 12-hour session limit. The trainer currently saves only at the end, so do not start the 50k run unless you can let it finish.

The notebook evaluates VQA accuracy and latency on held-out data. Description quality needs a separate fixed set and human review.


## 0. Configuration

- `TRAIN / VAL / TEST` — defaults to 20,000/1,000/1,000. For 50k, set `TRAIN = 50000`, `VAL = 2000`, `TEST = 2000` and use a fresh `VERSION`.
- `CUSTOM_EXAMPLES_FILE` — optional JSONL with your own image/question/answer examples; see section 3b. Leave as `None` to skip.
- `DTYPE` — `float16` works on common Kaggle and Colab GPUs. If loss becomes NaN, try `float32`.
- `SEED` — fixes the VQA train/validation/test image split.

Plan for roughly 4–7 hours at 20k. The 50k estimate is about 8–16 hours, depending on GPU and downloads; Kaggle documents a 12-hour notebook session maximum, so 50k may not finish in one session. The trainer saves only after all epochs, so 20k is the safer free run.


In [ ]:
VERSION = "v0.2-20k"
TRAIN, VAL, TEST = 20_000, 1_000, 1_000
CUSTOM_EXAMPLES_FILE = None  # e.g. "data/asksight_examples.jsonl"
DTYPE = "float16"
SEED = 0


## 1. Install (~2 min)

Clones the repo and installs the `asksight_model` package (torch, transformers, pillow). Re-run safe; no large downloads on Colab.

In [ ]:
import os
REPO = "https://github.com/Pranjal6955/AskSight.git"
BRANCH = "dev"
WORKSPACE = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR = os.path.join(WORKSPACE, "AskSight")
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone --branch {BRANCH} --depth 1 {REPO} {REPO_DIR}
%cd {REPO_DIR}/model
assert os.path.isdir("asksight_model"), f"model package missing in branch {BRANCH}"
# The clone is a snapshot of the pushed branch: a stale branch means the
# robustness module below is missing.
assert os.path.isfile("asksight_model/robustness.py"), (
    f"branch {BRANCH} is stale - push your model/ changes to GitHub before running"
)
!pip install -q -e ".[dev]"

## 2. Verify GPU

Asserts CUDA is available; prints the assigned GPU (expect `Tesla T4`). If it fails: **Runtime → Change runtime type → T4 GPU**, then re-run.

In [ ]:
!nvidia-smi -L
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU accelerator in the notebook Settings first"
import asksight_model
print("asksight_model", asksight_model.__name__, "ok")


## 3. Download VQA data

Downloads official VQA v2 questions/annotations plus the COCO images needed for this split. The split is deterministic and image-disjoint: images in train are kept separate from validation and test. The fingerprints are saved in `data/vqa/meta.json`.

For 50k, change the counts in cell 2 before running this cell. Actual counts can be slightly higher because the splitter keeps all questions for selected images together.


In [ ]:
%%time
!python -m asksight_model.data --out data/vqa --train {TRAIN} --val {VAL} --test {TEST} --seed {SEED}
!cat data/vqa/meta.json

## 3b. Add your own examples (optional)

Create a JSONL file with one JSON object per line. Put each image under `data/vqa/` and write its path relative to that folder. Each row needs a unique string `question_id` and `image_id`, plus `image`, `question`, and `answer`. Keep custom images separate from the VQA validation/test images. For a description example, use a question such as `Describe this photo` and write a grounded three-to-five-sentence answer. For a direct question, keep the target answer short.

Example lines (replace these paths and answers with real examples):

```json
{"question_id":"custom-001","image_id":"custom-image-001","image":"images/custom/menu.jpg","question":"What does the sign say?","answer":"The sign says ‘Fresh fruit’.","category":"reading_text"}
{"question_id":"custom-002","image_id":"custom-image-002","image":"images/custom/park.jpg","question":"Describe this photo.","answer":"A person is walking a brown dog along a paved path. Trees and a grassy area line the path. The sky appears overcast.","category":"description"}
```

For Kaggle, add a private Kaggle Dataset containing the JSONL and image files as a Notebook Input, then copy them into `data/vqa/` using the paths in the JSONL. For Colab, upload them into the runtime. Set `CUSTOM_EXAMPLES_FILE` in cell 2. These rows are added to training only, and their file fingerprint and count are recorded with the checkpoint. Review every answer for correctness; noisy or invented captions teach the model to make things up.


In [ ]:
# Optional custom examples are merged into train.jsonl only. Safe to re-run.
if CUSTOM_EXAMPLES_FILE:
    import hashlib
    import json
    from pathlib import Path

    examples_path = Path(CUSTOM_EXAMPLES_FILE)
    if not examples_path.is_file():
        raise FileNotFoundError(f"Upload your examples JSONL first: {examples_path}")

    data_dir = Path("data/vqa")
    train_path = data_dir / "train.jsonl"
    records = [json.loads(line) for line in train_path.read_text().splitlines() if line.strip()]
    records = [record for record in records if record.get("source") != "asksight_custom"]
    val_records = [json.loads(line) for line in (data_dir / "val.jsonl").read_text().splitlines() if line.strip()]
    test_records = [json.loads(line) for line in (data_dir / "test.jsonl").read_text().splitlines() if line.strip()]
    held_out_images = {str(r["image_id"]) for r in val_records + test_records}
    existing_ids = {str(r["question_id"]) for r in records}

    custom = [json.loads(line) for line in examples_path.read_text().splitlines() if line.strip()]
    for row_number, row in enumerate(custom, start=1):
        for field in ("question_id", "image_id", "image", "question", "answer"):
            if field not in row or not str(row[field]).strip():
                raise ValueError(f"custom row {row_number} needs a non-empty {field!r}")
        row["question_id"] = str(row["question_id"])
        row["image_id"] = str(row["image_id"])
        if row["image_id"] in held_out_images:
            raise ValueError(f"custom row {row_number} reuses a validation/test image_id")
        if row["question_id"] in existing_ids:
            raise ValueError(f"custom row {row_number} duplicates a question_id")
        if not (data_dir / row["image"]).is_file():
            raise FileNotFoundError(f"custom image not found: {data_dir / row['image']}")
        row["source"] = "asksight_custom"
        existing_ids.add(row["question_id"])
        records.append(row)

    train_path.write_text("".join(json.dumps(row) + "\n" for row in records))
    meta_path = data_dir / "meta.json"
    meta = json.loads(meta_path.read_text())
    meta.setdefault("custom_examples", {})
    meta["custom_examples"] = {
        "file": examples_path.name,
        "sha256": hashlib.sha256(examples_path.read_bytes()).hexdigest(),
        "added_to_train": len(custom),
    }
    meta["actual"]["train"] = len(records)
    meta["split_fingerprints"]["train"] = hashlib.sha256(
        ",".join(sorted(str(row["question_id"]) for row in records)).encode()
    ).hexdigest()
    meta_path.write_text(json.dumps(meta, indent=2) + "\n")
    print(f"Added {len(custom)} custom examples; train now has {len(records)} rows")
else:
    print("No custom examples selected; using VQA data only")


## 4. Train

The prompt now asks for short, direct answers to direct questions and three-to-five short sentences when the user asks for a photo description. The notebook uses that same prompt for training and serving. Description behavior also needs description examples; VQA-only training mostly teaches short answers.

Training runs for two epochs and saves only when complete. A 20k run is the recommended first attempt. For 50k, change the counts and version in cell 2, then rerun the data cell onward. Save the zip at the end. If loss becomes `nan`, try `DTYPE = "float32"`; on CUDA OOM, lower the batch size in `configs/default.yaml`.


In [ ]:
%%time
!python -m asksight_model.train --config configs/default.yaml --dtype {DTYPE} --version {VERSION}


## 5. Evaluate (~2 min)

VQA accuracy and latency are measured on the held-out test split and recorded in `metrics.json`. Custom description quality is not automatically scored; review the same fixed description questions for every candidate.


In [ ]:
%%time
!python -m asksight_model.evaluate --checkpoint checkpoints/{VERSION} --split test --data-dir data/vqa
import json
ev = json.load(open(f"checkpoints/{VERSION}/metrics.json"))["eval_test"]
print(f"\nVQA accuracy: {ev['vqa_accuracy']:.1%} | latency median {ev['latency_median_s']}s, p95 {ev['latency_p95_s']}s")

## 5b. Robustness (~3 min)

Re-runs the same held-out split under one realistic capture fault per condition — `blur`, `dark`, `obstructed` (a finger over the lens), `lowres` — and reports the accuracy drop against `clean`. This is the measured basis for the "please verify this" guidance the app shows for poor photos. The result lands in `metrics.json` under `eval_robustness`.


In [ ]:
%%time
!python -m asksight_model.robustness --checkpoint checkpoints/{VERSION} --split test --data-dir data/vqa


## 6. Download the checkpoint

Zips `checkpoints/<VERSION>/` and downloads it. Colab storage is ephemeral — the zip is the deliverable. It contains the trained connector weights, configs, metrics, and dataset provenance.

In [ ]:
!zip -qr {VERSION}-ckpt.zip checkpoints/{VERSION}
import os
if os.environ.get("COLAB_RELEASE_TAG"):  # real Colab runtime
    from google.colab import files
    files.download(f"{VERSION}-ckpt.zip")
else:
    zip_path = f"{VERSION}-ckpt.zip"
    if os.path.isdir("/kaggle/working"):
        import shutil
        output_path = f"/kaggle/working/{VERSION}-ckpt.zip"
        if os.path.abspath(zip_path) != os.path.abspath(output_path):
            shutil.copy2(zip_path, output_path)
        zip_path = output_path
    size = os.path.getsize(zip_path) / 1e6
    print(f"ready for the Kaggle Output panel: {os.path.abspath(zip_path)} ({size:.1f} MB)")

## 7. Serve the checkpoint (laptop / server)

```bash
cd ~/AskSight-checkpoints && unzip ~/Downloads/v0.2-20k-ckpt.zip
cd backend && source venv/bin/activate
pip install -e ../model
export ASKSIGHT_MODEL_CHECKPOINT=$HOME/AskSight-checkpoints/checkpoints/v0.2-20k
uvicorn app.main:app --reload   # /ask now returns real answers
```

For a larger experiment, set `TRAIN = 50000` and use a fresh `VERSION`. Compare on the same held-out data. Do not judge quality from answer length alone.